<a href="https://colab.research.google.com/github/lsgrep/serv/blob/main/notebooks/10_finetune_what_it_teaches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10 — A fine-tune, and what it actually taught the model

**The claim you should be able to make when you finish:** *"I've run the
experiment. Fine-tuning moved output-format compliance from a third to nearly
all of responses. On the same facts asked in different words, it moved recall
by a few points, while retrieval — same base model, no training — answered
them. I can show you the table."*

Every other lab in this repo, and every version of the retrieval-versus-fine-tuning
conversation, leans on one assertion:

> Fine-tuning teaches style and format. It does not install knowledge.

Asserting it is a position. Measuring it is a receipt, and this lab is small
enough to be a receipt: one 0.5B model, about 370 training rows, a few minutes
on a free T4.

### The experiment

Train **one** LoRA on **both** kinds of data at once — which is what a team
under deadline actually does when told to "fine-tune it on our docs" — then
measure the two axes separately:

| Axis | Test | What we expect |
|---|---|---|
| **Format** | strict JSON schema on unseen questions | large improvement — this is fine-tuning's home turf |
| **Knowledge, seen phrasings** | training questions, closed book | improvement, but it is memorisation |
| **Knowledge, held-out phrasings** | *same facts, different words*, closed book | the honest test |
| **Knowledge + retrieval** | held-out questions with the passage in context | the comparison that matters |
| **General capability** | unrelated trivia | the regression check nobody runs |

The held-out split is the whole design. Scoring well on phrasings you trained on
measures memorisation; only new wording of the same fact tells you whether
anything was learned.

In [ ]:
# Cell 1 — bootstrap. Idempotent; rerun freely after a disconnect.
REPO, BRANCH = "https://github.com/lsgrep/serv.git", "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib",
                "transformers", "peft", "trl", "datasets", "accelerate"], check=True)

import servlab
env = servlab.notebook_setup()

## 1. Should you be fine-tuning at all?

Walk this before writing a training script. Each step costs roughly 10x the
previous one in engineering, so earn it with eval data rather than enthusiasm.

1. **Prompting.** Free, instant, and the ceiling is higher than people assume.
2. **RAG.** When the failure is *missing knowledge* — the model was never handed
   the fact. Days of work.
3. **Fine-tuning.** When the failure is *style, format, or behaviour* at volume,
   after retrieval is proven. Weeks.
4. **Open weights, self-hosted.** Residency, edge, or unit economics at sustained
   scale. Quarters.

Legitimate reasons to fine-tune, all of which this lab's format task is a
miniature of:

* **A house output format** a parser depends on, where "right 90% of the time"
  means a retry path, latency and cost.
* **Distillation** — teaching a small cheap model to imitate a large one on a
  narrow task, which is a *cost* play measured in tokens saved.
* **Behaviour and tone** a prompt can only approximate, at a volume where prompt
  tokens are themselves a bill.
* **Refusal and safety boundaries** specific to a domain.

The reason that fails, and the one you will be asked about: *"the answers are
wrong, so let's train it on our documentation."* That is what the knowledge task
here is built to test.

## 2. The two datasets

Both are generated from the same five policy documents lab 8 uses for retrieval,
so the corpus is constant and only the *method* changes. That is the control
that makes the comparison mean something.

In [ ]:
from servlab import finetune as ft, rag

chunks, _ = rag.policy_corpus()
fmt_train = ft.format_examples(chunks, "train")
fmt_test = ft.format_examples(chunks, "test")
kn_train = ft.knowledge_examples("train")
kn_test = ft.knowledge_examples("test")

print(f"format    {len(fmt_train):>3} train  {len(fmt_test):>3} held-out")
print(f"knowledge {len(kn_train):>3} train  {len(kn_test):>3} held-out")
print(f"facts     {len(ft.FACTS)}")

In [ ]:
# The format task: question plus passage in, strict JSON out.
e = fmt_train[0]
print(e.system[:140], "...\n")
print(e.prompt[:260], "...\n")
print("target:", e.completion)

In [ ]:
# The knowledge task, and the trap that makes it an experiment.
fact = ft.FACTS[0]
print(f"fact: {fact.id} -> {fact.answer!r}\n")
print("TRAINED on these phrasings:")
for q in fact.train_questions:
    print(f"   {q}")
print("\nTESTED on these — same fact, different words, never seen in training:")
for q in fact.heldout_questions:
    print(f"   {q}")
print("\nIf the fine-tune scores well on the first set and poorly on the second,")
print("it memorised strings. That is the result to look for, not to avoid.")

## 3. Baselines, before any training

Four numbers to beat. Take them now — a baseline collected after you have seen
the fine-tune's results is not a baseline.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small on purpose: trains in minutes,
                                            # and weak enough that the format
                                            # lesson is visible rather than subtle
DTYPE = torch.bfloat16 if env.supports_bf16 else torch.float16

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE).to("cuda").eval()
print(f"{sum(p.numel() for p in base.parameters())/1e6:.0f}M parameters, {DTYPE}")

In [ ]:
def make_generator(model, max_new_tokens=64):
    """Greedy decoding, so every difference we measure is the weights."""
    def generate(prompt, system=""):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        return tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True)
    return generate

gen_base = make_generator(base)
print(gen_base("Say hello in five words.", "")[:120])

In [ ]:
# Baseline 1 — format compliance on held-out questions.
valid_ids = [c.id for c in chunks]
base_fmt_out = [gen_base(e.prompt, e.system) for e in fmt_test]
base_fmt = ft.format_rate(base_fmt_out, valid_ids)

for k, v in base_fmt.items():
    print(f"  {k:<18}{v if isinstance(v, int) else format(v, '.0%'):>8}")
print("\nA sample response:\n")
print(base_fmt_out[0][:300])

Look at the gap between `parses_anywhere` and `strict_json`. The model often
produces the right object *wrapped in conversation* — "Sure! Here's the JSON:" —
which a downstream parser rejects. That gap is exactly the kind of problem
fine-tuning is for, and exactly the kind that prompt engineering fixes only
most of the time.

In [ ]:
# Baselines 2 and 3 — knowledge, closed book, on both splits.
base_seen = ft.evaluate_knowledge(kn_train, gen_base, "base, seen phrasings")
base_held = ft.evaluate_knowledge(kn_test, gen_base, "base, held-out")
print(f"  seen phrasings : {base_seen['accuracy']:.0%}")
print(f"  held-out       : {base_held['accuracy']:.0%}")
print("\nBoth should be near zero — this company's policies are invented, so the")
print("base model has no way to know them. That is the point: we are about to")
print("try to install knowledge two different ways and compare.")

In [ ]:
# Baseline 4 — the same base model, with retrieval. No training whatsoever.
retriever = rag.HybridRetriever(chunks)

def make_rag_generator(model, k=2):
    inner = make_generator(model)
    def generate(prompt, system=""):
        hits = retriever.search(prompt, k=k)
        context = "\n\n".join(f"[{c.id}] {c.text}" for c, _ in hits)
        return inner(f"Context:\n{context}\n\nQuestion: {prompt}",
                     system or "Answer the question briefly using only the context.")
    return generate

base_rag = ft.evaluate_knowledge(kn_test, make_rag_generator(base), "base + retrieval")
print(f"  base + retrieval on held-out questions: {base_rag['accuracy']:.0%}")

## 4. Train the LoRA

One run, on both datasets mixed — the realistic version of "fine-tune it on our
docs". A few minutes on a T4.

The knobs and why these values: **rank 16** is enough for format learning and
small enough to train fast; **3 epochs** over a repeated set, because we want to
see what memorisation looks like rather than avoid it; **greedy eval** so nothing
is sampling noise.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

train_examples = ft.augment(fmt_train + kn_train, factor=8, seed=0)
rows = ft.to_dataset(train_examples, tokenizer=tok)
dataset = Dataset.from_list(rows)
print(f"{len(dataset)} training rows")
print(rows[0]["text"][:400])

In [ ]:
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
model = get_peft_model(base, peft_cfg)
model.print_trainable_parameters()

In [ ]:
from servlab.memory import gpu_memory_report, record_memory, reset_peak

reset_peak()
args = SFTConfig(
    output_dir="runs/lab10-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    max_length=512,
    fp16=not env.supports_bf16,
    bf16=env.supports_bf16,
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="no",
    report_to=[],
)

# Same instrumentation as lab 4: if it OOMs, you get a snapshot rather than a
# traceback. At this size it will not, which is itself worth noticing — the
# memory arithmetic said so before we started.
with record_memory("runs/lab10_oom.pickle"):
    trainer = SFTTrainer(model=model, args=args, train_dataset=dataset)
    trainer.train()

print("\n" + gpu_memory_report())

## 5. Measure the same four things again

Nothing about the evaluation changes — same prompts, same greedy decoding, same
scorers. Only the weights moved.

In [ ]:
tuned = trainer.model.eval()
gen_tuned = make_generator(tuned)

tuned_fmt_out = [gen_tuned(e.prompt, e.system) for e in fmt_test]
tuned_fmt = ft.format_rate(tuned_fmt_out, valid_ids)

print(f"{'metric':<20}{'base':>8}{'tuned':>8}")
for k in ("usable", "strict_json", "keys_ok", "confidence_ok", "citation_valid", "wrapped_in_prose"):
    print(f"{k:<20}{base_fmt[k]:>7.0%}{tuned_fmt[k]:>8.0%}")
print("\nA sample response after fine-tuning:\n")
print(tuned_fmt_out[0][:300])

In [ ]:
tuned_seen = ft.evaluate_knowledge(kn_train, gen_tuned, "tuned, seen phrasings")
tuned_held = ft.evaluate_knowledge(kn_test, gen_tuned, "tuned, held-out")
tuned_rag = ft.evaluate_knowledge(kn_test, make_rag_generator(tuned), "tuned + retrieval")

print(f"  seen phrasings   {base_seen['accuracy']:>6.0%} -> {tuned_seen['accuracy']:>4.0%}")
print(f"  held-out         {base_held['accuracy']:>6.0%} -> {tuned_held['accuracy']:>4.0%}")
print(f"  + retrieval      {base_rag['accuracy']:>6.0%} -> {tuned_rag['accuracy']:>4.0%}")

In [ ]:
# Read the actual held-out answers. The aggregate hides the interesting part:
# how the fine-tune fails when it does.
for r in tuned_held["rows"][:6]:
    mark = "ok  " if r["correct"] else "MISS"
    print(f"  {mark}  {r['question'][:56]:<58} expected {r['expected']!r:<24} got {r['got'][:44]!r}")

## 6. The regression check nobody runs

Narrow fine-tuning on a repetitive format is an extremely effective way to teach
a small model to emit that format **regardless of the question**. Two minutes to
check, and it is why "it works great on our task" and "it broke everything else"
so often ship together.

In [ ]:
cap_base = ft.capability_check(gen_base, name="base")
cap_tuned = ft.capability_check(gen_tuned, name="fine-tuned")
print(f"general capability: {cap_base['accuracy']:.0%} -> {cap_tuned['accuracy']:.0%}\n")
for b, t in zip(cap_base["rows"], cap_tuned["rows"]):
    print(f"  {b['prompt'][:38]:<40} base {b['got'][:24]!r:<28} tuned {t['got'][:24]!r}")

If the tuned column is emitting JSON at trivia questions, you have reproduced
catastrophic forgetting in a few minutes on a free GPU — and you now have a
first-hand answer to *"what are the risks of fine-tuning?"* that is not from a
blog post.

Mitigations, in the order you would reach for them: mix in general instruction
data (5-20% of the set), lower the rank, fewer epochs, a lower learning rate, or
keep the adapter switchable so the base behaviour is one flag away — which is
itself an argument for LoRA over full fine-tuning.

## 7. The table, and the sentence it earns

In [ ]:
results = [
    {"name": "base model", "format_usable": base_fmt["usable"],
     "known_phrasings": base_seen["accuracy"], "held_out": base_held["accuracy"],
     "with_retrieval": base_rag["accuracy"], "capability": cap_base["accuracy"]},
    {"name": "fine-tuned", "format_usable": tuned_fmt["usable"],
     "known_phrasings": tuned_seen["accuracy"], "held_out": tuned_held["accuracy"],
     "with_retrieval": tuned_rag["accuracy"], "capability": cap_tuned["accuracy"]},
]
print(ft.comparison_table(results))

In [ ]:
print(ft.verdict(
    fine_tuned_heldout=tuned_held["accuracy"],
    base_heldout=base_held["accuracy"],
    rag_heldout=base_rag["accuracy"],
    format_gain=tuned_fmt["usable"] - base_fmt["usable"],
))

In [ ]:
import matplotlib.pyplot as plt
from servlab.plots import use_style, SERIES

use_style()
labels = ["format\n(held-out)", "knowledge\n(seen phrasings)", "knowledge\n(held-out)",
          "knowledge\n(+ retrieval)"]
before = [base_fmt["usable"], base_seen["accuracy"], base_held["accuracy"], base_rag["accuracy"]]
after = [tuned_fmt["usable"], tuned_seen["accuracy"], tuned_held["accuracy"], tuned_rag["accuracy"]]

fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = range(len(labels))
ax.bar([i - 0.19 for i in x], [v * 100 for v in before], width=0.36,
       color=SERIES[1], label="base model")
ax.bar([i + 0.19 for i in x], [v * 100 for v in after], width=0.36,
       color=SERIES[0], label="after fine-tuning")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel("% correct / usable"); ax.set_ylim(0, 105)
ax.set_title("one fine-tune, two very different effects")
ax.legend(loc="upper right")
plt.show()

## 8. What it cost, versus what retrieval cost

The comparison an exec is actually making, and the one that makes the
recommendation concrete rather than doctrinal.

In [ ]:
from servlab import pricing as pr

train_minutes = trainer.state.log_history[-1].get("train_runtime", 0) / 60 if trainer.state.log_history else 0
gpu_hourly = pr.SelfHostPlan(nodes=1, gpus_per_node=1, usd_per_gpu_hour=2.50).compute_monthly / pr.HOURS_PER_MONTH

print(f"this fine-tune:  {train_minutes:.1f} min of GPU  ~= ${train_minutes/60*gpu_hourly:.2f}")
print("               + the engineering to build the dataset, which is the real cost")
print("               + an eval to prove it helped, which you need anyway")
print("               + a capability regression you now have to watch for")
print()
print("the retrieval path: no training at all, and it answered the held-out")
print(f"questions at {base_rag['accuracy']:.0%} on an untouched base model.")
print()
print("Scale that honestly for a real engagement: a 7B fine-tune on real data is")
print("days of data work and hundreds of dollars of compute per iteration — and")
print("iterations are what it takes. Retrieval is usually a week, and it is")
print("debuggable: when it is wrong you can see which passage it read.")

## 9. Fine-tune *and* retrieve

The false choice is the one to reject out loud. These are orthogonal: retrieval
supplies facts, fine-tuning supplies behaviour. The `tuned + retrieval` column
above is usually the best system in the table — right format, right facts.

The order matters operationally, though: **prove retrieval first**, because a
fine-tune built on top of broken retrieval bakes the wrong behaviour into
weights, and unlike a prompt you cannot revert it in an afternoon.

In [ ]:
# Save the adapter — a few MB, and the artifact worth keeping.
from servlab.env import mount_drive

dest = "/content/drive/MyDrive/servlab/lab10-adapter" if mount_drive() else "runs/lab10-adapter"
trainer.model.save_pretrained(dest)
tok.save_pretrained(dest)
print("adapter ->", dest)
print("\nLoRA adapters are switchable at serving time: one base model in memory,")
print("many adapters. vLLM serves them with --enable-lora, which also means a")
print("bad fine-tune is a config rollback rather than a redeploy.")

## What to be able to say afterwards

1. **The measured version of the claim.** Not "fine-tuning teaches style" — *"in
   my run, format compliance went from X to Y, held-out factual recall moved Z
   points, and retrieval on the untouched base model beat it."*
2. **Why the held-out split is the whole experiment.** Testing on phrasings you
   trained on measures memorisation, and most fine-tuning demos do exactly that.
3. **The legitimate reasons to fine-tune** — format, distillation for cost, tone,
   domain refusals — and that "the answers are wrong" is not one of them until
   retrieval has been ruled out.
4. **Catastrophic forgetting is real and cheap to check.** You have watched a
   model start answering trivia in JSON.
5. **They compose.** Retrieval for facts, fine-tuning for behaviour, and prove
   retrieval first because weights are harder to revert than prompts.

### The exec version

> "We ran it. Fine-tuning took our format compliance from about a third of
> responses to nearly all of them — that's real, and it's worth doing, because
> our parser needs valid output every time.
>
> On the factual questions it's a different story. It learned the exact questions
> we trained on. Asked the same things in different words, it was barely better
> than the untrained model — while the same base model with retrieval answered
> them, with a citation we can check.
>
> So I'd do both, in this order: retrieval first, because that's where the wrong
> answers were coming from, then the fine-tune for the output format. And I'd
> keep the fine-tune as a swappable adapter, so if it regresses we roll back a
> config rather than a deployment."

**Related:** [lab 4](04_qlora_oom_postmortem.ipynb) is the same machinery at 3B
scale where it OOMs; [lab 8](08_rag_and_evals.ipynb) is the retrieval half and
the triage that decides which of the two you need.